# Golden Config Comparison: Higgs USR vs Punctuated Inflation

Side-by-side comparison of the best configurations for both models:
- **Higgs USR**: $\phi_0=6.00$, $y_0=-0.538$, $N_*=37.3$, $\xi=15000$, $\lambda=0.13$
- **Punctuated Inflation**: $m=1.1323\times 10^{-7}$, $\lambda=3.3299\times 10^{-15}$, $N_*=77.2$ (perfect inflection)

Both produce $P_S(k)$ and $D_\ell^{TT}$ for $\ell \leq 30$, plotted against Planck 2018 low-$\ell$ TT data.

**Note:** Punctuated Inflation takes ~96 total e-folds and is expensive ($T_{max}=500000$, 100k bg steps). First run can take several minutes.

In [ ]:
import sys, os

import numpy as np
import matplotlib.pyplot as plt

from models import HiggsModel, PunctuatedInflationModel
from pspectrum_pipeline import run_pspectrum_pipeline, build_weighted_kgrid
from scripts.camb_wrapper import compute_cl_full_camb, compute_cl_camb_powerlaw
from scripts.planck_data import get_planck_data_asymmetric
from scripts.constants import T_cmb, As, k_pivot_phys, N_star_default, r_ls

## 1. Run Higgs USR Pipeline
$\phi_0=6.00$, $y_0=-0.538$, $N_*=37.3$

In [ ]:
higgs_model = HiggsModel(lam=0.13, xi=15000)
higgs_model.phi0 = 6.00
higgs_model.y0 = -0.538

print("Running Higgs USR pipeline...")
higgs_result = run_pspectrum_pipeline(
    model=higgs_model,
    phi0=6.00,
    y0=-0.538,
    N_star=37.3,
    k_min=1e-5,
    k_max=1.0,
    num_k=80,
    ms_steps=5000,
    normalize_to_As=True,
    save_outputs=True,
)
print(f"Higgs status: {higgs_result['status']} - {higgs_result['message']}")

## 2. Run Punctuated Inflation Pipeline
$m=1.1323\times 10^{-7}$, $\lambda=3.3299\times 10^{-15}$, $N_*=77.2$

**Derivation of $N_*=77.2$:**
1. The model produces $\sim 96.5$ total e-folds.
2. The USR peak occurs $\sim 2.9$ e-folds before the scale that exited at the pivot $k_*=0.05$.
3. To align the peak exactly at $k \approx 10^{-3}$ Mpc$^{-1}$ (as in Fig 4 of arXiv:1610.05776), we need $N_* \approx 77.2$.
4. This value shifts the spectral feature to larger physical scales (rightward) compared to the default $N_*=60$.

In [ ]:
punct_model = PunctuatedInflationModel(
    m=1.1323e-7,
    lam=3.3299e-15,
)

print("Running Punctuated Inflation pipeline...")
k_weighted = build_weighted_kgrid(1e-5, 1.0, k_pivot_phys, n_dense=60, n_outer=20)
punct_result = run_pspectrum_pipeline(
    model=punct_model,
    N_star=77.2,
    k_phys_grid=k_weighted,
    ms_steps=5000,
    normalize_to_As=True,
    save_outputs=True,
)
print(f"Punctuated status: {punct_result['status']} - {punct_result['message']}")

## 3. Check if cached results exist

If the cells above failed, check for existing cached JSON files in `outputs/simulations/pspectra/`.

In [ ]:
from pspectrum_pipeline import load_pspectrum
import glob

cached_files = sorted(glob.glob(os.path.join('..', 'outputs', 'simulations', 'pspectra', '*.json')))
if cached_files:
    print(f"Found {len(cached_files)} cached spectra:")
    for f in cached_files:
        meta = load_pspectrum(f)['metadata']
        print(f"  {os.path.basename(f)}  |  {meta.get('model','?'):30s}  phi={meta.get('phi0','?'):6s}  y={meta.get('y0','?'):6s}  N*={meta.get('N_star','?'):6s}")
else:
    print("No cached files found. Run the pipeline cells above first.")

## 4. Compute C_ell for Both Models

In [ ]:
ells_h, C_ell_h, _, _ = compute_cl_full_camb(higgs_result, ell_max=30)
D_ell_h = ells_h * (ells_h + 1) / (2 * np.pi) * C_ell_h * T_cmb**2 * 1e12

ells_p, C_ell_p, _, _ = compute_cl_full_camb(punct_result, ell_max=30)
D_ell_p = ells_p * (ells_p + 1) / (2 * np.pi) * C_ell_p * T_cmb**2 * 1e12

# LCDM power-law baseline
ells_pl, C_ell_pl, _, _ = compute_cl_camb_powerlaw(ell_max=30, ns=0.965)
D_ell_pl = ells_pl * (ells_pl + 1) / (2 * np.pi) * C_ell_pl * T_cmb**2 * 1e12

# Planck data
ells_planck, D_planck, D_err_lower, D_err_upper = get_planck_data_asymmetric()

print(f"C_ell computed: Higgs ({len(ells_h)} ells), Punctuated ({len(ells_p)} ells)")
print(f"Planck data: {len(ells_planck)} bins")

## 5. Side-by-Side $P_S(k)$ Comparison

In [ ]:
k_h = higgs_result['k_phys']
ps_h = higgs_result['P_S']
k_p = punct_result['k_phys']
ps_p = punct_result['P_S']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# ---- Higgs ----
ax1.loglog(k_h, ps_h, 'b-', linewidth=2.5, label='Higgs USR')
ax1.axvline(k_pivot_phys, color='gray', ls='--', alpha=0.5, label=f'pivot $k_*={k_pivot_phys}$')
ax1.set_xlabel(r'$k$ [Mpc$^{-1}$]', fontsize=14)
ax1.set_ylabel(r'$\mathcal{P}_\mathcal{S}(k)$', fontsize=14)
ax1.set_title('Higgs USR ($\\phi_0=6.00, y_0=-0.538, N_*=37.3$)', fontsize=12)
ax1.legend(fontsize=11)
ax1.grid(True, which='major', alpha=0.3)
ax1.grid(True, which='minor', alpha=0.15)

# ---- Punctuated ----
ax2.loglog(k_p, ps_p, 'r-', linewidth=2.5, label='Punctuated Inflation')
ax2.axvline(k_pivot_phys, color='gray', ls='--', alpha=0.5, label=f'pivot $k_*={k_pivot_phys}$')
ax2.set_xlabel(r'$k$ [Mpc$^{-1}$]', fontsize=14)
ax2.set_ylabel(r'$\mathcal{P}_\mathcal{S}(k)$', fontsize=14)
ax2.set_title('Punctuated Inflation ($N_*=75.6$, weighted grid)', fontsize=12)
ax2.legend(fontsize=11)
ax2.grid(True, which='major', alpha=0.3)
ax2.grid(True, which='minor', alpha=0.15)

fig.tight_layout()
from scripts.plotting import get_path, make_filename
fig.savefig(get_path("powerloss", "ps_comparison_golden.png"), dpi=200, bbox_inches='tight')
print('Saved: outputs/plots/powerloss/golden_comparison_ps.png')
plt.show()

## 6. Side-by-Side $D_\ell^{TT}$ Comparison

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5.5))

for ax in (ax1, ax2):
    ax.axvspan(1.5, 5.5, color='#e6e6fa', alpha=0.5)
    ax.text(3.5, 200, r'$\ell \leq 5$', rotation=90, color='#7b68ee',
            verticalalignment='center', horizontalalignment='center', fontsize=14, alpha=0.8)
    ax.errorbar(ells_planck, D_planck, yerr=[D_err_upper, D_err_lower],
                fmt='o', color='#34495e', capsize=3, elinewidth=1.5, markersize=7,
                label=r'Planck 2018 low-$\ell$ TT')
    ax.plot(ells_pl, D_ell_pl, color='#7f8c8d', lw=2.5, ls='--',
            label=r'LCDM ($n_s=0.965$)')
    ax.set_yscale('log')
    ax.set_xlim(1.5, 30)
    ax.set_ylim(80, 2500)
    ax.set_xlabel(r'$\ell$', fontsize=14)
    ax.set_ylabel(r'$D_\ell^{\,TT}$ [$\mu$K$^2$]', fontsize=14)
    ax.grid(True, which='major', alpha=0.3)
    ax.grid(True, which='minor', alpha=0.15)
    ax.legend(fontsize=10, loc='lower right')

ax1.plot(ells_h, D_ell_h, color='#d62728', lw=2.5, label='Higgs USR')
ax1.set_title('Higgs USR ($\\phi_0=6.00, y_0=-0.538$)', fontsize=13)

ax2.plot(ells_p, D_ell_p, color='#d62728', lw=2.5, label='Punctuated')
ax2.set_title('Punctuated Inflation ($N_*=75.6$, weighted)', fontsize=13)

fig.tight_layout()
fig.savefig(get_path("powerloss", "dell_comparison_golden.png"), dpi=200, bbox_inches='tight')
print('Saved: outputs/plots/powerloss/golden_comparison_dell.png')
plt.show()

## 7. Overlaid $D_\ell^{TT}$ Comparison (Single Panel)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

ax.axvspan(1.5, 5.5, color='#e6e6fa', alpha=0.5)
ax.text(3.5, 200, r'$\ell \leq 5$', rotation=90, color='#7b68ee',
        verticalalignment='center', horizontalalignment='center', fontsize=14, alpha=0.8)

ax.errorbar(ells_planck, D_planck, yerr=[D_err_lower, D_err_upper],
            fmt='o', color='#34495e', capsize=3, elinewidth=1.5, markersize=7,
            label=r'Planck 2018 low-$\ell$ TT')
ax.plot(ells_pl, D_ell_pl, color='#7f8c8d', lw=2.5, ls='--',
        label=r'LCDM ($n_s=0.965$)')
ax.plot(ells_h, D_ell_h, color='#1f77b4', lw=2.5, label='Higgs USR')
ax.plot(ells_p, D_ell_p, color='#ff7f0e', lw=2.5, label='Punctuated Inflation', zorder=3)

ax.set_yscale('log')
ax.set_xlim(1.5, 30)
ax.set_ylim(80, 2500)
ax.set_xlabel(r'$\ell$', fontsize=14)
ax.set_ylabel(r'$D_\ell^{\,TT}$ [$\mu$K$^2$]', fontsize=14)
ax.grid(True, which='major', alpha=0.3)
ax.grid(True, which='minor', alpha=0.15)
ax.legend(fontsize=11, loc='lower right')

fig.tight_layout()
fig.savefig(get_path("powerloss", "dell_comparison_overlaid.png"), dpi=200, bbox_inches='tight')
print('Saved: outputs/plots/powerloss/golden_comparison_overlaid.png')
plt.show()

## 8. Summary Statistics

In [ ]:
def chi2_model(D_model, D_data, err_lower, err_upper):
    chi2 = 0
    for i, ell_val in enumerate(ells_planck):
        idx = np.argmin(np.abs(ells_h - ell_val))
        residual = D_model[idx] - D_data[i]
        sigma = err_upper[i] if residual > 0 else err_lower[i]
        chi2 += (residual / sigma)**2
    return chi2

chi2_h = chi2_model(D_ell_h, D_planck, D_err_lower, D_err_upper)
chi2_p = chi2_model(D_ell_p, D_planck, D_err_lower, D_err_upper)
chi2_lcdm = chi2_model(D_ell_pl, D_planck, D_err_lower, D_err_upper)

print(f"{'Model':<25} {'chi2':>8} {'Delta chi2':>12}")
print(f"{'-'*45}")
print(f"{'LCDM (ns=0.965)':<25} {chi2_lcdm:>8.2f} {'0.00':>12}")
print(f"{'Higgs USR':<25} {chi2_h:>8.2f} {chi2_h - chi2_lcdm:>12.2f}")
print(f"{'Punctuated Inflation':<25} {chi2_p:>8.2f} {chi2_p - chi2_lcdm:>12.2f}")